# 106 — TCGA Methylation Download Validation

## Objective

This notebook validates the completeness and byte-level integrity of the downloaded TCGA primary-tumor DNA methylation cohort frozen by:

- `104_tcga_methylation_coverage_assessment.ipynb`
- `105_tcga_methylation_cohort_freeze.ipynb`

The operational acquisition cohort contains all source-complete HM27 and HM450 SeSAMe methylation beta-value files. EPICv2 files were excluded by the platform acquisition decision documented in notebook 104.

## Expected frozen cohort

The frozen GDC manifest defines:

- 10,897 methylation beta-value payloads
- 10,703 aliquots
- 10,656 samples
- 10,610 cases
- 33 TCGA projects
- 2 retained platforms: HM27 and HM450
- 106.735678 GiB of expected payload data

## Validation criteria

The download is considered complete and valid only if:

1. Every manifest file ID has a corresponding local directory.
2. Every expected payload filename is present in its corresponding file-ID directory.
3. No expected payload is missing.
4. Every payload has the exact byte size recorded in the frozen manifest.
5. No partial-download or temporary payload files remain.
6. Every payload MD5 checksum matches the frozen GDC manifest.
7. Auxiliary GDC annotation files are inventoried and parsed separately from the manifest payloads.
8. The observed methylation platforms remain restricted to HM27 and HM450 according to the frozen file index.
9. A minimal content-level inspection confirms that downloaded payloads are readable SeSAMe methylation beta-value text files.

## Inputs

- `config/manifests/tcga_methylation/gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt`
- `config/manifests/tcga_methylation/gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv`
- `config/manifests/tcga_methylation/gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json`
- `data/raw/tcga/methylation/`

## Scope

This notebook validates acquisition integrity and preserves GDC annotation information. It does not yet perform methylation quality control, probe filtering, sample selection, platform harmonization, normalization, or biological analysis.

A successful MD5 validation establishes that the local payloads are byte-identical to the files specified by the frozen GDC manifest. Biological and assay-level quality control will be addressed in downstream notebooks.

In [1]:
# =============================================================================
# 106 — TCGA Methylation Download Validation
# Imports, project-root detection, and input paths
# =============================================================================

import hashlib
import json
import sys
import time
from pathlib import Path

import pandas as pd


# -----------------------------------------------------------------------------
# Detect project root
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the pancancer-epigenetics project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# Project utilities
# -----------------------------------------------------------------------------

from src.utils.paths import Paths
from src.utils.file_checks import (
    calculate_sha256,
    get_file_size_mb,
    to_project_relative_posix_path,
)


# -----------------------------------------------------------------------------
# Frozen cohort inputs
# -----------------------------------------------------------------------------

METHYLATION_MANIFEST_DIR = (
    Paths.config / "manifests" / "tcga_methylation"
)

FROZEN_MANIFEST_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt"
)

FROZEN_FILE_INDEX_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv"
)

FROZEN_COHORT_METADATA_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json"
)


# -----------------------------------------------------------------------------
# Local download directory
# -----------------------------------------------------------------------------

METHYLATION_DOWNLOAD_DIR = (
    Paths.tcga / "methylation"
)


# -----------------------------------------------------------------------------
# Expected frozen-cohort properties
# -----------------------------------------------------------------------------

EXPECTED_FILE_COUNT = 10_897
EXPECTED_TOTAL_SIZE_BYTES = 114_606_561_259

EXPECTED_PLATFORMS = {
    "Illumina Human Methylation 27",
    "Illumina Human Methylation 450",
}


print(f"Project root:          {PROJECT_ROOT}")
print(f"Frozen manifest:       {FROZEN_MANIFEST_PATH}")
print(f"Frozen file index:     {FROZEN_FILE_INDEX_PATH}")
print(f"Frozen cohort record:  {FROZEN_COHORT_METADATA_PATH}")
print(f"Download directory:    {METHYLATION_DOWNLOAD_DIR}")
print()
print(f"Expected payloads:     {EXPECTED_FILE_COUNT:,}")
print(
    f"Expected payload size: "
    f"{EXPECTED_TOTAL_SIZE_BYTES / (1024**3):,.6f} GiB"
)

Project root:          C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics
Frozen manifest:       C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt
Frozen file index:     C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv
Frozen cohort record:  C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json
Download directory:    C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\raw\tcga\methylation

Expected payloads:     10,897
Expected payload size: 106.735678 GiB


In [2]:
# =============================================================================
# Load and validate frozen cohort artifacts
# =============================================================================

required_input_paths = {
    "manifest": FROZEN_MANIFEST_PATH,
    "file_index": FROZEN_FILE_INDEX_PATH,
    "cohort_freeze": FROZEN_COHORT_METADATA_PATH,
}

missing_inputs = [
    path
    for path in required_input_paths.values()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required frozen-cohort inputs:\n"
        + "\n".join(str(path) for path in missing_inputs)
    )

non_file_inputs = [
    path
    for path in required_input_paths.values()
    if not path.is_file()
]

if non_file_inputs:
    raise ValueError(
        "The following frozen-cohort inputs are not files:\n"
        + "\n".join(str(path) for path in non_file_inputs)
    )

if not METHYLATION_DOWNLOAD_DIR.exists():
    raise FileNotFoundError(
        f"Download directory not found: {METHYLATION_DOWNLOAD_DIR}"
    )

if not METHYLATION_DOWNLOAD_DIR.is_dir():
    raise NotADirectoryError(
        f"Download path is not a directory: "
        f"{METHYLATION_DOWNLOAD_DIR}"
    )


# -----------------------------------------------------------------------------
# Load frozen artifacts
# -----------------------------------------------------------------------------

manifest_df = pd.read_csv(
    FROZEN_MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": "string",
        "filename": "string",
        "md5": "string",
        "state": "string",
    },
)

manifest_df["size"] = pd.to_numeric(
    manifest_df["size"],
    errors="raise",
).astype("int64")

file_index_df = pd.read_csv(
    FROZEN_FILE_INDEX_PATH,
    sep="\t",
    dtype={
        "file_id": "string",
        "file_name": "string",
        "md5": "string",
        "state": "string",
        "platform": "string",
    },
)

file_index_df["file_size_bytes"] = pd.to_numeric(
    file_index_df["file_size_bytes"],
    errors="raise",
).astype("int64")

with FROZEN_COHORT_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    cohort_freeze_record = json.load(handle)


# -----------------------------------------------------------------------------
# Validate required columns
# -----------------------------------------------------------------------------

required_manifest_columns = {
    "id",
    "filename",
    "md5",
    "size",
    "state",
}

required_file_index_columns = {
    "file_id",
    "file_name",
    "md5",
    "file_size_bytes",
    "state",
    "platform",
    "project_id",
    "case_uuid",
    "sample_submitter_id",
    "aliquot_uuid",
}

missing_manifest_columns = (
    required_manifest_columns - set(manifest_df.columns)
)

missing_file_index_columns = (
    required_file_index_columns - set(file_index_df.columns)
)

if missing_manifest_columns:
    raise KeyError(
        "Missing manifest columns: "
        + ", ".join(sorted(missing_manifest_columns))
    )

if missing_file_index_columns:
    raise KeyError(
        "Missing file-index columns: "
        + ", ".join(sorted(missing_file_index_columns))
    )


# -----------------------------------------------------------------------------
# Validate row counts and identifiers
# -----------------------------------------------------------------------------

if len(manifest_df) != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Unexpected manifest row count: {len(manifest_df):,}"
    )

if len(file_index_df) != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Unexpected file-index row count: {len(file_index_df):,}"
    )

if manifest_df["id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in frozen manifest.")

if file_index_df["file_id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in frozen file index.")

manifest_file_ids = set(manifest_df["id"])
file_index_file_ids = set(file_index_df["file_id"])

if manifest_file_ids != file_index_file_ids:
    raise ValueError(
        "Manifest and file index contain different file-ID sets."
    )


# -----------------------------------------------------------------------------
# Validate manifest against canonical file index
# -----------------------------------------------------------------------------

manifest_index_comparison = (
    manifest_df
    .rename(
        columns={
            "id": "file_id",
            "filename": "manifest_file_name",
            "md5": "manifest_md5",
            "size": "manifest_file_size_bytes",
            "state": "manifest_state",
        }
    )
    .merge(
        file_index_df[
            [
                "file_id",
                "file_name",
                "md5",
                "file_size_bytes",
                "state",
            ]
        ],
        on="file_id",
        how="inner",
        validate="one_to_one",
    )
)

artifact_mismatches = {
    "file_name": int(
        manifest_index_comparison["manifest_file_name"]
        .ne(manifest_index_comparison["file_name"])
        .sum()
    ),
    "md5": int(
        manifest_index_comparison["manifest_md5"]
        .str.lower()
        .ne(manifest_index_comparison["md5"].str.lower())
        .sum()
    ),
    "file_size_bytes": int(
        manifest_index_comparison["manifest_file_size_bytes"]
        .ne(manifest_index_comparison["file_size_bytes"])
        .sum()
    ),
    "state": int(
        manifest_index_comparison["manifest_state"]
        .ne(manifest_index_comparison["state"])
        .sum()
    ),
}

if any(artifact_mismatches.values()):
    raise ValueError(
        f"Manifest-versus-file-index mismatches: "
        f"{artifact_mismatches}"
    )


# -----------------------------------------------------------------------------
# Validate frozen cohort properties
# -----------------------------------------------------------------------------

observed_total_size_bytes = int(manifest_df["size"].sum())
observed_platforms = set(file_index_df["platform"].dropna().unique())
observed_states = set(manifest_df["state"].dropna().unique())

if observed_total_size_bytes != EXPECTED_TOTAL_SIZE_BYTES:
    raise ValueError(
        "Unexpected frozen payload size:\n"
        f"Observed: {observed_total_size_bytes:,} bytes\n"
        f"Expected: {EXPECTED_TOTAL_SIZE_BYTES:,} bytes"
    )

if observed_platforms != EXPECTED_PLATFORMS:
    raise ValueError(
        f"Unexpected frozen platforms: "
        f"{sorted(observed_platforms)}"
    )

if observed_states != {"released"}:
    raise ValueError(
        f"Unexpected manifest states: {sorted(observed_states)}"
    )


# -----------------------------------------------------------------------------
# Validate cohort-freeze record and artifact hashes
# -----------------------------------------------------------------------------

recorded_counts = cohort_freeze_record["frozen_cohort_counts"]

if recorded_counts["n_files"] != EXPECTED_FILE_COUNT:
    raise ValueError(
        "Cohort-freeze JSON contains an unexpected file count."
    )

if (
    recorded_counts["total_file_size_bytes"]
    != EXPECTED_TOTAL_SIZE_BYTES
):
    raise ValueError(
        "Cohort-freeze JSON contains an unexpected payload size."
    )

recorded_platforms = set(
    cohort_freeze_record[
        "platform_acquisition_decision"
    ]["included_platforms"]
)

if recorded_platforms != EXPECTED_PLATFORMS:
    raise ValueError(
        "Cohort-freeze JSON contains an unexpected platform decision."
    )

current_artifact_sha256 = {
    "manifest": calculate_sha256(FROZEN_MANIFEST_PATH),
    "file_index": calculate_sha256(FROZEN_FILE_INDEX_PATH),
}

recorded_artifact_sha256 = {
    label: cohort_freeze_record["versioned_outputs"][label]["sha256"]
    for label in current_artifact_sha256
}

artifact_hash_matches = {
    label: (
        current_artifact_sha256[label]
        == recorded_artifact_sha256[label]
    )
    for label in current_artifact_sha256
}

if not all(artifact_hash_matches.values()):
    raise ValueError(
        f"Frozen artifact SHA-256 mismatch: "
        f"{artifact_hash_matches}"
    )


print("Frozen methylation cohort artifacts loaded and validated.")
print(f"Manifest rows:       {len(manifest_df):,}")
print(f"File-index rows:     {len(file_index_df):,}")
print(f"Unique file IDs:     {len(manifest_file_ids):,}")
print(f"Platforms:           {sorted(observed_platforms)}")
print(
    f"Expected payload:    "
    f"{observed_total_size_bytes / (1024**3):,.6f} GiB"
)
print(f"Manifest SHA-256:    {artifact_hash_matches['manifest']}")
print(f"File-index SHA-256:  {artifact_hash_matches['file_index']}")
print(f"Download directory:  {METHYLATION_DOWNLOAD_DIR}")

Frozen methylation cohort artifacts loaded and validated.
Manifest rows:       10,897
File-index rows:     10,897
Unique file IDs:     10,897
Platforms:           ['Illumina Human Methylation 27', 'Illumina Human Methylation 450']
Expected payload:    106.735678 GiB
Manifest SHA-256:    True
File-index SHA-256:  True
Download directory:  C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\raw\tcga\methylation


In [3]:
# =============================================================================
# Inventory local methylation download structure
# =============================================================================

top_level_entries = list(METHYLATION_DOWNLOAD_DIR.iterdir())

top_level_directories = [
    path
    for path in top_level_entries
    if path.is_dir()
]

top_level_files = [
    path
    for path in top_level_entries
    if path.is_file()
]

other_top_level_entries = [
    path
    for path in top_level_entries
    if not path.is_dir() and not path.is_file()
]


# -----------------------------------------------------------------------------
# Inventory direct files, nested directories, and nested files
# -----------------------------------------------------------------------------

direct_file_rows = []
nested_directory_rows = []
nested_file_rows = []
other_folder_entry_rows = []

for folder_number, folder_path in enumerate(
    sorted(top_level_directories),
    start=1,
):
    folder_id = folder_path.name

    try:
        folder_entries = list(folder_path.iterdir())
    except OSError as error:
        raise OSError(
            f"Could not inspect download folder: {folder_path}"
        ) from error

    for entry_path in folder_entries:
        if entry_path.is_file():
            direct_file_rows.append(
                {
                    "folder_id": folder_id,
                    "file_name": entry_path.name,
                    "file_path": entry_path,
                    "file_size_bytes": entry_path.stat().st_size,
                    "extension": entry_path.suffix.lower(),
                }
            )

        elif entry_path.is_dir():
            nested_directory_rows.append(
                {
                    "parent_folder_id": folder_id,
                    "directory_name": entry_path.name,
                    "directory_path": entry_path,
                }
            )

            for nested_path in entry_path.rglob("*"):
                if nested_path.is_file():
                    nested_file_rows.append(
                        {
                            "parent_folder_id": folder_id,
                            "parent_directory": entry_path.name,
                            "relative_path": nested_path.relative_to(
                                folder_path
                            ).as_posix(),
                            "file_path": nested_path,
                            "file_size_bytes": nested_path.stat().st_size,
                        }
                    )

        else:
            other_folder_entry_rows.append(
                {
                    "folder_id": folder_id,
                    "entry_name": entry_path.name,
                    "entry_path": entry_path,
                }
            )

    if folder_number % 2_000 == 0:
        print(
            f"Inventoried {folder_number:,} / "
            f"{len(top_level_directories):,} directories"
        )


# -----------------------------------------------------------------------------
# Build inventory tables
# -----------------------------------------------------------------------------

direct_files_df = pd.DataFrame(
    direct_file_rows,
    columns=[
        "folder_id",
        "file_name",
        "file_path",
        "file_size_bytes",
        "extension",
    ],
)

nested_directories_df = pd.DataFrame(
    nested_directory_rows,
    columns=[
        "parent_folder_id",
        "directory_name",
        "directory_path",
    ],
)

nested_files_df = pd.DataFrame(
    nested_file_rows,
    columns=[
        "parent_folder_id",
        "parent_directory",
        "relative_path",
        "file_path",
        "file_size_bytes",
    ],
)

other_folder_entries_df = pd.DataFrame(
    other_folder_entry_rows,
    columns=[
        "folder_id",
        "entry_name",
        "entry_path",
    ],
)

observed_folder_ids = {
    path.name
    for path in top_level_directories
}


# -----------------------------------------------------------------------------
# Per-folder distributions
# -----------------------------------------------------------------------------

direct_files_per_folder = (
    direct_files_df
    .groupby("folder_id")
    .size()
    .reindex(sorted(observed_folder_ids), fill_value=0)
)

direct_file_count_distribution = (
    direct_files_per_folder
    .value_counts()
    .sort_index()
    .rename_axis("direct_files_per_folder")
    .reset_index(name="folder_count")
)

nested_directories_per_folder = (
    nested_directories_df
    .groupby("parent_folder_id")
    .size()
    .reindex(sorted(observed_folder_ids), fill_value=0)
)

nested_directory_count_distribution = (
    nested_directories_per_folder
    .value_counts()
    .sort_index()
    .rename_axis("nested_directories_per_folder")
    .reset_index(name="folder_count")
)


# -----------------------------------------------------------------------------
# Inventory summary
# -----------------------------------------------------------------------------

observed_direct_size_bytes = int(
    direct_files_df["file_size_bytes"].sum()
)

observed_nested_size_bytes = int(
    nested_files_df["file_size_bytes"].sum()
)

print()
print("Local methylation download structure inventoried.")
print(f"Top-level directories:    {len(top_level_directories):,}")
print(f"Top-level files:          {len(top_level_files):,}")
print(f"Direct files:             {len(direct_files_df):,}")
print(f"Nested directories:       {len(nested_directories_df):,}")
print(f"Nested files:             {len(nested_files_df):,}")
print(f"Other top-level entries:  {len(other_top_level_entries):,}")
print(f"Other folder entries:     {len(other_folder_entries_df):,}")
print(
    f"Observed direct size:     "
    f"{observed_direct_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Observed nested size:     "
    f"{observed_nested_size_bytes / (1024**2):,.3f} MiB"
)

print()
print("Direct files per top-level directory:")
print(direct_file_count_distribution)

print()
print("Nested directories per top-level directory:")
print(nested_directory_count_distribution)

if not nested_directories_df.empty:
    print()
    print("Nested-directory names:")
    print(
        nested_directories_df["directory_name"]
        .value_counts()
        .rename_axis("directory_name")
        .reset_index(name="directory_count")
    )

Inventoried 2,000 / 10,897 directories
Inventoried 4,000 / 10,897 directories
Inventoried 6,000 / 10,897 directories
Inventoried 8,000 / 10,897 directories
Inventoried 10,000 / 10,897 directories

Local methylation download structure inventoried.
Top-level directories:    10,897
Top-level files:          0
Direct files:             12,142
Nested directories:       9,102
Nested files:             9,102
Other top-level entries:  0
Other folder entries:     0
Observed direct size:     106.736174 GiB
Observed nested size:     11.268 MiB

Direct files per top-level directory:
   direct_files_per_folder  folder_count
0                        1          9652
1                        2          1245

Nested directories per top-level directory:
   nested_directories_per_folder  folder_count
0                              0          1795
1                              1          9102

Nested-directory names:
  directory_name  directory_count
0           logs             9102


In [4]:
# =============================================================================
# Compare frozen manifest against local download
# =============================================================================

# -----------------------------------------------------------------------------
# Expected payload table
# -----------------------------------------------------------------------------

expected_payloads_df = (
    manifest_df
    .rename(
        columns={
            "id": "folder_id",
            "filename": "file_name",
            "md5": "expected_md5",
            "size": "expected_size_bytes",
            "state": "expected_state",
        }
    )
    .copy()
)

expected_payloads_df["expected_path"] = (
    expected_payloads_df.apply(
        lambda row: (
            METHYLATION_DOWNLOAD_DIR
            / row["folder_id"]
            / row["file_name"]
        ),
        axis=1,
    )
)

expected_folder_ids = set(expected_payloads_df["folder_id"])


# -----------------------------------------------------------------------------
# Folder-level comparison
# -----------------------------------------------------------------------------

missing_folder_ids = sorted(
    expected_folder_ids - observed_folder_ids
)

unexpected_folder_ids = sorted(
    observed_folder_ids - expected_folder_ids
)


# -----------------------------------------------------------------------------
# Payload-level comparison
# -----------------------------------------------------------------------------

observed_payload_columns = (
    direct_files_df[
        [
            "folder_id",
            "file_name",
            "file_path",
            "file_size_bytes",
        ]
    ]
    .rename(
        columns={
            "file_path": "local_path",
            "file_size_bytes": "local_size_bytes",
        }
    )
)

payload_comparison_df = expected_payloads_df.merge(
    observed_payload_columns,
    on=["folder_id", "file_name"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

missing_payloads_df = (
    payload_comparison_df.loc[
        payload_comparison_df["_merge"] == "left_only"
    ]
    .copy()
)

present_payloads_df = (
    payload_comparison_df.loc[
        payload_comparison_df["_merge"] == "both"
    ]
    .copy()
)

size_mismatches_df = (
    present_payloads_df.loc[
        present_payloads_df["expected_size_bytes"]
        != present_payloads_df["local_size_bytes"]
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Identify auxiliary direct files
# -----------------------------------------------------------------------------

expected_payload_keys = expected_payloads_df[
    ["folder_id", "file_name"]
]

extra_direct_files_df = (
    direct_files_df
    .merge(
        expected_payload_keys,
        on=["folder_id", "file_name"],
        how="left",
        indicator=True,
    )
    .loc[lambda frame: frame["_merge"] == "left_only"]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

annotation_files_df = (
    extra_direct_files_df.loc[
        extra_direct_files_df["file_name"] == "annotations.txt"
    ]
    .copy()
)

unexpected_auxiliary_files_df = (
    extra_direct_files_df.loc[
        extra_direct_files_df["file_name"] != "annotations.txt"
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Detect partial-like or temporary files
# -----------------------------------------------------------------------------

partial_like_suffixes = (
    ".partial",
    ".part",
    ".tmp",
    ".download",
    ".incomplete",
)

direct_partial_like_files_df = (
    direct_files_df.loc[
        direct_files_df["file_name"]
        .str.lower()
        .str.endswith(partial_like_suffixes)
    ]
    .copy()
)

nested_partial_like_files_df = (
    nested_files_df.loc[
        nested_files_df["relative_path"]
        .str.lower()
        .str.endswith(partial_like_suffixes)
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Size summaries
# -----------------------------------------------------------------------------

expected_payload_size_bytes = int(
    expected_payloads_df["expected_size_bytes"].sum()
)

observed_payload_size_bytes = int(
    present_payloads_df["local_size_bytes"].sum()
)

extra_direct_size_bytes = int(
    extra_direct_files_df["file_size_bytes"].sum()
)

annotation_size_bytes = int(
    annotation_files_df["file_size_bytes"].sum()
)


# -----------------------------------------------------------------------------
# Structural validation summary
# -----------------------------------------------------------------------------

print("Expected-versus-observed comparison completed.")
print(f"Expected folders:          {len(expected_folder_ids):,}")
print(f"Observed folders:          {len(observed_folder_ids):,}")
print(f"Missing folders:           {len(missing_folder_ids):,}")
print(f"Unexpected folders:        {len(unexpected_folder_ids):,}")
print(
    f"Expected payloads present: "
    f"{len(present_payloads_df):,}"
)
print(f"Missing payloads:          {len(missing_payloads_df):,}")
print(f"Size mismatches:           {len(size_mismatches_df):,}")
print(f"Extra direct files:        {len(extra_direct_files_df):,}")
print(f"Annotation files:          {len(annotation_files_df):,}")
print(
    f"Unexpected auxiliary files:"
    f" {len(unexpected_auxiliary_files_df):,}"
)
print(
    f"Direct partial-like files: "
    f"{len(direct_partial_like_files_df):,}"
)
print(
    f"Nested partial-like files: "
    f"{len(nested_partial_like_files_df):,}"
)
print(
    f"Expected payload size:     "
    f"{expected_payload_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Observed payload size:     "
    f"{observed_payload_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Annotation size:           "
    f"{annotation_size_bytes / (1024**2):,.3f} MiB"
)


# -----------------------------------------------------------------------------
# Auxiliary-file summary
# -----------------------------------------------------------------------------

if not extra_direct_files_df.empty:
    print()
    print("Extra direct files by exact filename:")
    print(
        extra_direct_files_df
        .groupby("file_name", as_index=False)
        .agg(
            file_count=("file_name", "size"),
            total_size_bytes=("file_size_bytes", "sum"),
        )
        .assign(
            total_size_mib=lambda frame: (
                frame["total_size_bytes"] / (1024**2)
            )
        )
        .sort_values(
            ["file_count", "file_name"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


# -----------------------------------------------------------------------------
# Fail closed if acquisition structure is incomplete
# -----------------------------------------------------------------------------

structural_failures = {
    "missing_folders": len(missing_folder_ids),
    "unexpected_folders": len(unexpected_folder_ids),
    "missing_payloads": len(missing_payloads_df),
    "size_mismatches": len(size_mismatches_df),
    "unexpected_auxiliary_files": len(
        unexpected_auxiliary_files_df
    ),
    "direct_partial_like_files": len(
        direct_partial_like_files_df
    ),
    "top_level_files": len(top_level_files),
    "other_top_level_entries": len(other_top_level_entries),
    "other_folder_entries": len(other_folder_entries_df),
}

nonzero_structural_failures = {
    label: count
    for label, count in structural_failures.items()
    if count != 0
}

if observed_payload_size_bytes != expected_payload_size_bytes:
    nonzero_structural_failures[
        "payload_size_difference_bytes"
    ] = (
        observed_payload_size_bytes
        - expected_payload_size_bytes
    )

if nonzero_structural_failures:
    raise RuntimeError(
        "Local download failed structural validation:\n"
        + json.dumps(
            nonzero_structural_failures,
            indent=2,
        )
    )

print()
print("Local download passed structural and size validation.")

Expected-versus-observed comparison completed.
Expected folders:          10,897
Observed folders:          10,897
Missing folders:           0
Unexpected folders:        0
Expected payloads present: 10,897
Missing payloads:          0
Size mismatches:           0
Extra direct files:        1,245
Annotation files:          1,245
Unexpected auxiliary files: 0
Direct partial-like files: 0
Nested partial-like files: 0
Expected payload size:     106.735678 GiB
Observed payload size:     106.735678 GiB
Annotation size:           0.508 MiB

Extra direct files by exact filename:
         file_name  file_count  total_size_bytes  total_size_mib
0  annotations.txt        1245            533151        0.508452

Local download passed structural and size validation.


In [5]:
# =============================================================================
# Validate payload MD5 checksums against the frozen GDC manifest
# =============================================================================

MD5_CHUNK_SIZE_BYTES = 8 * 1024 * 1024
MD5_PROGRESS_INTERVAL = 500


def calculate_md5(
    file_path,
    chunk_size_bytes=MD5_CHUNK_SIZE_BYTES,
):
    """Calculate the MD5 checksum of a local file."""

    md5_digest = hashlib.md5()

    with file_path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size_bytes)

            if not chunk:
                break

            md5_digest.update(chunk)

    return md5_digest.hexdigest()


# -----------------------------------------------------------------------------
# Prepare checksum-validation input
# -----------------------------------------------------------------------------

md5_input_df = (
    present_payloads_df[
        [
            "folder_id",
            "file_name",
            "expected_md5",
            "local_path",
            "local_size_bytes",
        ]
    ]
    .sort_values(["folder_id", "file_name"])
    .reset_index(drop=True)
)

if len(md5_input_df) != EXPECTED_FILE_COUNT:
    raise RuntimeError(
        "MD5 validation input does not contain the complete "
        "frozen cohort:\n"
        f"Observed payloads: {len(md5_input_df):,}\n"
        f"Expected payloads: {EXPECTED_FILE_COUNT:,}"
    )

invalid_expected_md5 = (
    ~md5_input_df["expected_md5"]
    .str.fullmatch(r"[0-9A-Fa-f]{32}", na=False)
)

if invalid_expected_md5.any():
    raise ValueError(
        "The frozen manifest contains invalid or missing MD5 values."
    )


# -----------------------------------------------------------------------------
# Calculate and compare checksums
# -----------------------------------------------------------------------------

md5_validation_rows = []

total_payloads = len(md5_input_df)
total_size_bytes = int(
    md5_input_df["local_size_bytes"].sum()
)

successfully_hashed_size_bytes = 0
validation_start_time = time.perf_counter()

for file_number, row in enumerate(
    md5_input_df.itertuples(index=False),
    start=1,
):
    local_path = Path(row.local_path)
    expected_md5 = str(row.expected_md5).lower()

    observed_md5 = None
    read_error = None

    try:
        observed_md5 = calculate_md5(local_path)
        successfully_hashed_size_bytes += int(
            row.local_size_bytes
        )

    except OSError as error:
        read_error = (
            f"{type(error).__name__}: {error}"
        )

    md5_matches = (
        read_error is None
        and observed_md5 == expected_md5
    )

    md5_validation_rows.append(
        {
            "folder_id": row.folder_id,
            "file_name": row.file_name,
            "local_path": local_path,
            "expected_md5": expected_md5,
            "observed_md5": observed_md5,
            "md5_matches": md5_matches,
            "read_error": read_error,
            "file_size_bytes": int(row.local_size_bytes),
        }
    )

    if (
        file_number % MD5_PROGRESS_INTERVAL == 0
        or file_number == total_payloads
    ):
        elapsed_seconds = (
            time.perf_counter()
            - validation_start_time
        )

        throughput_mib_per_second = (
            successfully_hashed_size_bytes
            / (1024**2)
            / elapsed_seconds
            if elapsed_seconds > 0
            else 0.0
        )

        print(
            f"Validated {file_number:,} / "
            f"{total_payloads:,} payloads | "
            f"{successfully_hashed_size_bytes / (1024**3):,.3f} GiB | "
            f"{throughput_mib_per_second:,.1f} MiB/s"
        )


# -----------------------------------------------------------------------------
# Build validation table
# -----------------------------------------------------------------------------

md5_validation_df = pd.DataFrame(md5_validation_rows)

md5_read_errors_df = (
    md5_validation_df.loc[
        md5_validation_df["read_error"].notna()
    ]
    .copy()
)

md5_mismatches_df = (
    md5_validation_df.loc[
        md5_validation_df["read_error"].isna()
        & ~md5_validation_df["md5_matches"]
    ]
    .copy()
)

md5_matches_df = (
    md5_validation_df.loc[
        md5_validation_df["md5_matches"]
    ]
    .copy()
)

validation_elapsed_seconds = (
    time.perf_counter()
    - validation_start_time
)

overall_throughput_mib_per_second = (
    successfully_hashed_size_bytes
    / (1024**2)
    / validation_elapsed_seconds
    if validation_elapsed_seconds > 0
    else 0.0
)


# -----------------------------------------------------------------------------
# Validation summary
# -----------------------------------------------------------------------------

print()
print("Payload MD5 validation completed.")
print(f"Expected payloads:       {EXPECTED_FILE_COUNT:,}")
print(f"Checksums calculated:    {len(md5_matches_df) + len(md5_mismatches_df):,}")
print(f"Checksum matches:        {len(md5_matches_df):,}")
print(f"Checksum mismatches:     {len(md5_mismatches_df):,}")
print(f"Read errors:             {len(md5_read_errors_df):,}")
print(
    f"Successfully hashed:     "
    f"{successfully_hashed_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Elapsed time:            "
    f"{validation_elapsed_seconds / 60:,.2f} minutes"
)
print(
    f"Average throughput:      "
    f"{overall_throughput_mib_per_second:,.1f} MiB/s"
)


# -----------------------------------------------------------------------------
# Show failures before stopping
# -----------------------------------------------------------------------------

if not md5_read_errors_df.empty:
    print()
    print("Payloads with read errors:")
    print(
        md5_read_errors_df[
            [
                "folder_id",
                "file_name",
                "read_error",
            ]
        ].head(20)
    )

if not md5_mismatches_df.empty:
    print()
    print("Payloads with checksum mismatches:")
    print(
        md5_mismatches_df[
            [
                "folder_id",
                "file_name",
                "expected_md5",
                "observed_md5",
            ]
        ].head(20)
    )


# -----------------------------------------------------------------------------
# Fail closed if byte-level validation is unsuccessful
# -----------------------------------------------------------------------------

md5_failures = {
    "read_errors": len(md5_read_errors_df),
    "checksum_mismatches": len(md5_mismatches_df),
}

if (
    successfully_hashed_size_bytes
    != total_size_bytes
):
    md5_failures["unhashed_size_bytes"] = (
        total_size_bytes
        - successfully_hashed_size_bytes
    )

nonzero_md5_failures = {
    label: value
    for label, value in md5_failures.items()
    if value != 0
}

if nonzero_md5_failures:
    raise RuntimeError(
        "Local payloads failed MD5 validation:\n"
        + json.dumps(
            nonzero_md5_failures,
            indent=2,
        )
    )

print()
print(
    "All local methylation payloads match "
    "the frozen GDC manifest MD5 checksums."
)

Validated 500 / 10,897 payloads | 4.804 GiB | 209.3 MiB/s
Validated 1,000 / 10,897 payloads | 9.695 GiB | 220.7 MiB/s
Validated 1,500 / 10,897 payloads | 14.357 GiB | 221.7 MiB/s
Validated 2,000 / 10,897 payloads | 19.237 GiB | 225.3 MiB/s
Validated 2,500 / 10,897 payloads | 24.280 GiB | 228.2 MiB/s
Validated 3,000 / 10,897 payloads | 29.196 GiB | 228.9 MiB/s
Validated 3,500 / 10,897 payloads | 34.147 GiB | 229.7 MiB/s
Validated 4,000 / 10,897 payloads | 39.236 GiB | 231.0 MiB/s
Validated 4,500 / 10,897 payloads | 44.151 GiB | 231.2 MiB/s
Validated 5,000 / 10,897 payloads | 49.021 GiB | 231.7 MiB/s
Validated 5,500 / 10,897 payloads | 53.835 GiB | 232.4 MiB/s
Validated 6,000 / 10,897 payloads | 58.668 GiB | 231.9 MiB/s
Validated 6,500 / 10,897 payloads | 63.779 GiB | 230.8 MiB/s
Validated 7,000 / 10,897 payloads | 68.703 GiB | 231.2 MiB/s
Validated 7,500 / 10,897 payloads | 73.457 GiB | 230.6 MiB/s
Validated 8,000 / 10,897 payloads | 78.494 GiB | 230.5 MiB/s
Validated 8,500 / 10,897 pay

In [6]:
# =============================================================================
# Inspect GDC annotation-file structure before parsing
# =============================================================================

if annotation_files_df.empty:
    raise RuntimeError(
        "No GDC annotations.txt files were found."
    )


# -----------------------------------------------------------------------------
# Read and characterize every annotation file
# -----------------------------------------------------------------------------

annotation_structure_rows = []

for row in (
    annotation_files_df
    .sort_values(["folder_id", "file_name"])
    .itertuples(index=False)
):
    annotation_path = Path(row.file_path)

    try:
        annotation_text = annotation_path.read_text(
            encoding="utf-8-sig"
        )
        read_error = None

    except (OSError, UnicodeError) as error:
        annotation_text = ""
        read_error = (
            f"{type(error).__name__}: {error}"
        )

    annotation_lines = annotation_text.splitlines()
    nonempty_lines = [
        line
        for line in annotation_lines
        if line.strip()
    ]

    first_nonempty_line = (
        nonempty_lines[0]
        if nonempty_lines
        else None
    )

    annotation_structure_rows.append(
        {
            "folder_id": row.folder_id,
            "file_path": annotation_path,
            "file_size_bytes": int(row.file_size_bytes),
            "line_count": len(annotation_lines),
            "nonempty_line_count": len(nonempty_lines),
            "first_nonempty_line": first_nonempty_line,
            "first_line_tab_count": (
                first_nonempty_line.count("\t")
                if first_nonempty_line is not None
                else 0
            ),
            "read_error": read_error,
        }
    )

annotation_structure_df = pd.DataFrame(
    annotation_structure_rows
)

annotation_read_errors_df = (
    annotation_structure_df.loc[
        annotation_structure_df["read_error"].notna()
    ]
    .copy()
)

empty_annotation_files_df = (
    annotation_structure_df.loc[
        annotation_structure_df["nonempty_line_count"] == 0
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Structural summary
# -----------------------------------------------------------------------------

print("GDC annotation-file inspection completed.")
print(
    f"Annotation files inspected: "
    f"{len(annotation_structure_df):,}"
)
print(
    f"Readable annotation files:  "
    f"{annotation_structure_df['read_error'].isna().sum():,}"
)
print(
    f"Read errors:                "
    f"{len(annotation_read_errors_df):,}"
)
print(
    f"Empty annotation files:     "
    f"{len(empty_annotation_files_df):,}"
)

print()
print("Line-count distribution:")
print(
    annotation_structure_df[
        "nonempty_line_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("nonempty_line_count")
    .reset_index(name="file_count")
)

print()
print("First-line tab-count distribution:")
print(
    annotation_structure_df[
        "first_line_tab_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("first_line_tab_count")
    .reset_index(name="file_count")
)


# -----------------------------------------------------------------------------
# Display representative raw contents
# -----------------------------------------------------------------------------

representative_annotation_paths = (
    annotation_structure_df.loc[
        annotation_structure_df["read_error"].isna()
        & (
            annotation_structure_df[
                "nonempty_line_count"
            ] > 0
        )
    ]
    .drop_duplicates(
        subset=[
            "nonempty_line_count",
            "first_nonempty_line",
        ]
    )
    .sort_values(
        [
            "nonempty_line_count",
            "folder_id",
        ]
    )
    ["file_path"]
    .head(3)
    .tolist()
)

print()
print("Representative annotation-file contents:")

for annotation_number, annotation_path in enumerate(
    representative_annotation_paths,
    start=1,
):
    print()
    print("=" * 80)
    print(
        f"Representative annotation file "
        f"{annotation_number}"
    )
    print(annotation_path)
    print("-" * 80)

    annotation_text = Path(annotation_path).read_text(
        encoding="utf-8-sig"
    )

    print(annotation_text)

GDC annotation-file inspection completed.
Annotation files inspected: 1,245
Readable annotation files:  1,245
Read errors:                0
Empty annotation files:     0

Line-count distribution:
   nonempty_line_count  file_count
0                    2        1062
1                    3         158
2                    4          22
3                    5           3

First-line tab-count distribution:
   first_line_tab_count  file_count
0                     8        1245

Representative annotation-file contents:

Representative annotation file 1
C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\raw\tcga\methylation\0014d611-dd6f-45da-ade0-56acfca30fe0\annotations.txt
--------------------------------------------------------------------------------
id	submitter_id	entity_type	entity_id	category	classification	created_datetime	status	notes
c106ab8e-44d4-5a74-8771-ed0e3c36c6a4	12164	case	300fa3a7-3c42-4edc-8992-400188e762b4	Alternate sample pipeline	Notification	20

In [7]:
# =============================================================================
# Parse and audit GDC annotation records
# =============================================================================

EXPECTED_ANNOTATION_COLUMNS = [
    "id",
    "submitter_id",
    "entity_type",
    "entity_id",
    "category",
    "classification",
    "created_datetime",
    "status",
    "notes",
]


# -----------------------------------------------------------------------------
# Parse every annotations.txt file
# -----------------------------------------------------------------------------

parsed_annotation_frames = []

for row in (
    annotation_files_df
    .sort_values(["folder_id", "file_name"])
    .itertuples(index=False)
):
    annotation_path = Path(row.file_path)

    try:
        annotation_frame = pd.read_csv(
            annotation_path,
            sep="\t",
            dtype="string",
            keep_default_na=False,
        )

    except Exception as error:
        raise RuntimeError(
            f"Could not parse annotation file: "
            f"{annotation_path}"
        ) from error

    observed_columns = annotation_frame.columns.tolist()

    if observed_columns != EXPECTED_ANNOTATION_COLUMNS:
        raise ValueError(
            "Unexpected annotation-file schema:\n"
            f"File: {annotation_path}\n"
            f"Observed: {observed_columns}\n"
            f"Expected: {EXPECTED_ANNOTATION_COLUMNS}"
        )

    if annotation_frame.empty:
        raise ValueError(
            f"Annotation file contains no records: "
            f"{annotation_path}"
        )

    annotation_frame.insert(
        0,
        "file_id",
        str(row.folder_id),
    )

    annotation_frame["annotation_path"] = annotation_path

    parsed_annotation_frames.append(annotation_frame)


annotation_associations_df = pd.concat(
    parsed_annotation_frames,
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Validate file-to-annotation associations
# -----------------------------------------------------------------------------

duplicate_annotation_associations_df = (
    annotation_associations_df.loc[
        annotation_associations_df.duplicated(
            subset=["file_id", "id"],
            keep=False,
        )
    ]
    .sort_values(["file_id", "id"])
    .copy()
)

annotation_metadata_columns = [
    column
    for column in EXPECTED_ANNOTATION_COLUMNS
    if column != "id"
]

annotation_metadata_variation = (
    annotation_associations_df
    .groupby("id", dropna=False)[
        annotation_metadata_columns
    ]
    .nunique(dropna=False)
)

conflicting_annotation_ids = (
    annotation_metadata_variation
    .gt(1)
    .any(axis=1)
)

conflicting_annotation_ids = set(
    annotation_metadata_variation.index[
        conflicting_annotation_ids
    ]
)

conflicting_annotations_df = (
    annotation_associations_df.loc[
        annotation_associations_df["id"].isin(
            conflicting_annotation_ids
        )
    ]
    .sort_values(["id", "file_id"])
    .copy()
)


# -----------------------------------------------------------------------------
# Build unique annotation table
# -----------------------------------------------------------------------------

unique_annotations_df = (
    annotation_associations_df[
        EXPECTED_ANNOTATION_COLUMNS
    ]
    .drop_duplicates()
    .sort_values("id")
    .reset_index(drop=True)
)

if unique_annotations_df["id"].duplicated().any():
    raise RuntimeError(
        "Annotation IDs remain duplicated after exact-row "
        "deduplication, indicating conflicting metadata."
    )


# -----------------------------------------------------------------------------
# Check required identifier fields
# -----------------------------------------------------------------------------

required_nonempty_annotation_fields = [
    "id",
    "entity_type",
    "entity_id",
    "category",
    "classification",
    "created_datetime",
    "status",
]

empty_required_field_counts = {
    column: int(
        unique_annotations_df[column]
        .str.strip()
        .eq("")
        .sum()
    )
    for column in required_nonempty_annotation_fields
}

nonzero_empty_required_fields = {
    column: count
    for column, count in empty_required_field_counts.items()
    if count != 0
}


# -----------------------------------------------------------------------------
# Summary tables
# -----------------------------------------------------------------------------

annotations_per_file_distribution = (
    annotation_associations_df
    .groupby("file_id")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("annotations_per_file")
    .reset_index(name="file_count")
)

files_per_annotation_distribution = (
    annotation_associations_df
    .groupby("id")["file_id"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("files_per_annotation")
    .reset_index(name="annotation_count")
)

entity_type_summary = (
    unique_annotations_df["entity_type"]
    .value_counts()
    .rename_axis("entity_type")
    .reset_index(name="annotation_count")
)

classification_summary = (
    unique_annotations_df["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="annotation_count")
)

status_summary = (
    unique_annotations_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="annotation_count")
)

category_summary = (
    unique_annotations_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="annotation_count")
)


# -----------------------------------------------------------------------------
# Annotation audit summary
# -----------------------------------------------------------------------------

print("GDC annotation records parsed and audited.")
print(
    f"Annotation files parsed:          "
    f"{len(annotation_files_df):,}"
)
print(
    f"File-to-annotation associations:  "
    f"{len(annotation_associations_df):,}"
)
print(
    f"Unique annotation IDs:            "
    f"{len(unique_annotations_df):,}"
)
print(
    f"Unique annotated entity IDs:      "
    f"{unique_annotations_df['entity_id'].nunique():,}"
)
print(
    f"Duplicate file-annotation links:  "
    f"{len(duplicate_annotation_associations_df):,}"
)
print(
    f"Conflicting annotation IDs:       "
    f"{len(conflicting_annotation_ids):,}"
)
print(
    f"Empty required fields:            "
    f"{sum(nonzero_empty_required_fields.values()):,}"
)

print()
print("Annotations per payload file:")
print(annotations_per_file_distribution)

print()
print("Payload files per unique annotation:")
print(files_per_annotation_distribution)

print()
print("Entity types:")
print(entity_type_summary)

print()
print("Classifications:")
print(classification_summary)

print()
print("Statuses:")
print(status_summary)

print()
print("Annotation categories:")
print(category_summary)


# -----------------------------------------------------------------------------
# Fail closed on malformed or contradictory annotation metadata
# -----------------------------------------------------------------------------

annotation_failures = {
    "duplicate_file_annotation_links": len(
        duplicate_annotation_associations_df
    ),
    "conflicting_annotation_ids": len(
        conflicting_annotation_ids
    ),
    "empty_required_fields": sum(
        nonzero_empty_required_fields.values()
    ),
}

nonzero_annotation_failures = {
    label: count
    for label, count in annotation_failures.items()
    if count != 0
}

if nonzero_annotation_failures:
    raise RuntimeError(
        "GDC annotation metadata failed validation:\n"
        + json.dumps(
            nonzero_annotation_failures,
            indent=2,
        )
    )

print()
print(
    "All GDC annotation records were parsed without "
    "schema errors, duplicate links, or conflicting metadata."
)

GDC annotation records parsed and audited.
Annotation files parsed:          1,245
File-to-annotation associations:  1,456
Unique annotation IDs:            1,242
Unique annotated entity IDs:      1,051
Duplicate file-annotation links:  0
Conflicting annotation IDs:       0
Empty required fields:            0

Annotations per payload file:
   annotations_per_file  file_count
0                     1        1062
1                     2         158
2                     3          22
3                     4           3

Payload files per unique annotation:
   files_per_annotation  annotation_count
0                     1              1038
1                     2               196
2                     3                 6
3                     4                 2

Entity types:
  entity_type  annotation_count
0        case              1141
1     analyte                87
2     aliquot                 9
3      sample                 4
4     portion                 1

Classifications:
  cla

In [8]:
# =============================================================================
# Inspect representative HM27 and HM450 payload contents
# =============================================================================

payload_location_df = (
    present_payloads_df[
        [
            "folder_id",
            "file_name",
            "local_path",
            "local_size_bytes",
        ]
    ]
    .rename(columns={"folder_id": "file_id"})
)

payload_format_inventory_df = (
    file_index_df[
        [
            "file_id",
            "file_name",
            "platform",
            "project_id",
        ]
    ]
    .merge(
        payload_location_df,
        on=["file_id", "file_name"],
        how="inner",
        validate="one_to_one",
    )
)

if len(payload_format_inventory_df) != EXPECTED_FILE_COUNT:
    raise RuntimeError(
        "Could not associate every frozen file-index record "
        "with its local payload."
    )


# -----------------------------------------------------------------------------
# Select one deterministic example from each retained platform
# -----------------------------------------------------------------------------

representative_payloads_df = (
    payload_format_inventory_df
    .sort_values(
        [
            "platform",
            "project_id",
            "file_id",
        ]
    )
    .groupby(
        "platform",
        sort=True,
        group_keys=False,
    )
    .head(1)
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# Display the first lines without hiding delimiter characters
# -----------------------------------------------------------------------------

print("Representative SeSAMe payload inspection:")

for row in representative_payloads_df.itertuples(
    index=False
):
    payload_path = Path(row.local_path)

    print()
    print("=" * 80)
    print(f"Platform:   {row.platform}")
    print(f"Project:    {row.project_id}")
    print(f"File ID:    {row.file_id}")
    print(f"File name:  {row.file_name}")
    print(
        f"File size:  "
        f"{row.local_size_bytes / (1024**2):,.3f} MiB"
    )
    print("-" * 80)

    try:
        with payload_path.open(
            "r",
            encoding="utf-8-sig",
            newline="",
        ) as handle:
            for line_number in range(1, 7):
                line = handle.readline()

                if line == "":
                    break

                print(
                    f"{line_number}: "
                    f"{line.rstrip(chr(10) + chr(13))!r}"
                )

    except (OSError, UnicodeError) as error:
        raise RuntimeError(
            f"Could not read representative payload: "
            f"{payload_path}"
        ) from error

Representative SeSAMe payload inspection:

Platform:   Illumina Human Methylation 27
Project:    TCGA-BRCA
File ID:    00c656d5-afb4-475e-aada-81ed728c42e4
File name:  ee2026e1-ac8e-420b-ab7f-9aa12b334875.methylation_array.sesame.level3betas.txt
File size:  0.735 MiB
--------------------------------------------------------------------------------
1: 'cg00000292\t0.865310277919857'
2: 'cg00002426\t0.907321823701685'
3: 'cg00003994\t0.344858777456292'
4: 'cg00005847\t0.781467999556097'
5: 'cg00006414\t0.0383588011479012'
6: 'cg00007981\t0.0350513348754637'

Platform:   Illumina Human Methylation 450
Project:    TCGA-ACC
File ID:    0222503f-764e-46d5-b7f6-b44a129c6da0
File name:  0cb9dee0-a437-4504-b874-3a60fcb121a7.methylation_array.sesame.level3betas.txt
File size:  12.561 MiB
--------------------------------------------------------------------------------
1: 'cg00000029\t0.574366288235562'
2: 'cg00000108\t0.972728733269739'
3: 'cg00000109\t0.941612885964817'
4: 'cg00000165\t0.11970287

In [9]:
# =============================================================================
# Perform cohort-wide minimal SeSAMe payload content validation
# =============================================================================

import math


MISSING_BETA_TOKENS = {
    "",
    "na",
    "nan",
    "null",
}


def read_record_after_offset(
    handle,
    byte_offset,
):
    """Read the first complete record after a byte offset."""

    handle.seek(byte_offset)

    if byte_offset > 0:
        handle.readline()

    return handle.readline()


def read_last_nonempty_record(
    handle,
    file_size_bytes,
):
    """Read the last nonempty record from a binary text file."""

    if file_size_bytes == 0:
        return b""

    window_size = min(
        file_size_bytes,
        64 * 1024,
    )

    while True:
        handle.seek(
            file_size_bytes - window_size
        )

        final_block = handle.read(window_size)

        contains_line_break = (
            b"\n" in final_block
            or b"\r" in final_block
        )

        if (
            window_size == file_size_bytes
            or contains_line_break
        ):
            nonempty_lines = [
                line
                for line in final_block.splitlines()
                if line.strip()
            ]

            return (
                nonempty_lines[-1]
                if nonempty_lines
                else b""
            )

        window_size = min(
            file_size_bytes,
            window_size * 2,
        )


def parse_sampled_beta_record(raw_record):
    """Validate one sampled SeSAMe beta-value record."""

    result = {
        "probe_id": None,
        "probe_prefix": None,
        "beta_token": None,
        "beta_value": None,
        "beta_is_missing": False,
        "validation_error": None,
    }

    if not raw_record:
        result["validation_error"] = "missing_record"
        return result

    try:
        record_text = raw_record.decode("utf-8")

    except UnicodeDecodeError:
        result["validation_error"] = "utf8_decode_error"
        return result

    record_text = record_text.rstrip("\r\n")

    if not record_text.strip():
        result["validation_error"] = "empty_record"
        return result

    record_fields = record_text.split("\t")

    if len(record_fields) != 2:
        result["validation_error"] = (
            f"unexpected_field_count:{len(record_fields)}"
        )
        return result

    probe_id = record_fields[0].strip()
    beta_token = record_fields[1].strip()

    result["probe_id"] = probe_id
    result["beta_token"] = beta_token

    if not probe_id:
        result["validation_error"] = "empty_probe_id"
        return result

    probe_prefix_characters = []

    for character in probe_id:
        if not character.isalpha():
            break

        probe_prefix_characters.append(character)

    result["probe_prefix"] = "".join(
        probe_prefix_characters
    )

    if beta_token.lower() in MISSING_BETA_TOKENS:
        result["beta_is_missing"] = True
        return result

    try:
        beta_value = float(beta_token)

    except ValueError:
        result["validation_error"] = (
            "nonnumeric_beta_value"
        )
        return result

    if not math.isfinite(beta_value):
        result["validation_error"] = (
            "nonfinite_beta_value"
        )
        return result

    if not 0.0 <= beta_value <= 1.0:
        result["validation_error"] = (
            "beta_outside_unit_interval"
        )
        return result

    result["beta_value"] = beta_value

    return result


# -----------------------------------------------------------------------------
# Sample five positions from every retained payload
# -----------------------------------------------------------------------------

sampled_payload_record_rows = []

for file_number, row in enumerate(
    payload_format_inventory_df
    .sort_values(["platform", "file_id"])
    .itertuples(index=False),
    start=1,
):
    payload_path = Path(row.local_path)
    file_size_bytes = int(row.local_size_bytes)

    offset_positions = [
        ("first_record", 0),
        ("quarter_position", file_size_bytes // 4),
        ("middle_position", file_size_bytes // 2),
        (
            "three_quarter_position",
            3 * file_size_bytes // 4,
        ),
    ]

    try:
        with payload_path.open("rb") as handle:
            sampled_records = [
                (
                    position_label,
                    byte_offset,
                    read_record_after_offset(
                        handle,
                        byte_offset,
                    ),
                )
                for (
                    position_label,
                    byte_offset,
                ) in offset_positions
            ]

            sampled_records.append(
                (
                    "last_record",
                    file_size_bytes,
                    read_last_nonempty_record(
                        handle,
                        file_size_bytes,
                    ),
                )
            )

    except OSError as error:
        sampled_payload_record_rows.append(
            {
                "file_id": row.file_id,
                "file_name": row.file_name,
                "platform": row.platform,
                "project_id": row.project_id,
                "sample_position": "file_open",
                "byte_offset": None,
                "probe_id": None,
                "probe_prefix": None,
                "beta_token": None,
                "beta_value": None,
                "beta_is_missing": False,
                "validation_error": (
                    f"{type(error).__name__}: {error}"
                ),
            }
        )

        continue

    for (
        position_label,
        byte_offset,
        raw_record,
    ) in sampled_records:
        parsed_record = parse_sampled_beta_record(
            raw_record
        )

        sampled_payload_record_rows.append(
            {
                "file_id": row.file_id,
                "file_name": row.file_name,
                "platform": row.platform,
                "project_id": row.project_id,
                "sample_position": position_label,
                "byte_offset": byte_offset,
                **parsed_record,
            }
        )

    if (
        file_number % 2_000 == 0
        or file_number == EXPECTED_FILE_COUNT
    ):
        print(
            f"Inspected {file_number:,} / "
            f"{EXPECTED_FILE_COUNT:,} payloads"
        )


sampled_payload_records_df = pd.DataFrame(
    sampled_payload_record_rows
)


# -----------------------------------------------------------------------------
# Validate sampling coverage and record structure
# -----------------------------------------------------------------------------

sample_counts_per_payload = (
    sampled_payload_records_df
    .groupby("file_id")
    .size()
    .reindex(
        payload_format_inventory_df["file_id"],
        fill_value=0,
    )
)

unexpected_sample_counts_df = (
    sample_counts_per_payload.loc[
        sample_counts_per_payload != 5
    ]
    .rename("sample_count")
    .reset_index()
)

content_validation_errors_df = (
    sampled_payload_records_df.loc[
        sampled_payload_records_df[
            "validation_error"
        ].notna()
    ]
    .copy()
)

missing_sampled_beta_values_df = (
    sampled_payload_records_df.loc[
        sampled_payload_records_df[
            "beta_is_missing"
        ]
    ]
    .copy()
)

valid_numeric_sampled_records_df = (
    sampled_payload_records_df.loc[
        sampled_payload_records_df[
            "validation_error"
        ].isna()
        & ~sampled_payload_records_df[
            "beta_is_missing"
        ]
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Content-validation summaries
# -----------------------------------------------------------------------------

probe_prefix_summary = (
    sampled_payload_records_df.loc[
        sampled_payload_records_df[
            "validation_error"
        ].isna()
    ]
    ["probe_prefix"]
    .value_counts(dropna=False)
    .rename_axis("probe_prefix")
    .reset_index(name="sampled_record_count")
)

sampled_beta_range_by_platform = (
    valid_numeric_sampled_records_df
    .groupby("platform", as_index=False)
    .agg(
        sampled_records=("beta_value", "size"),
        minimum_beta=("beta_value", "min"),
        maximum_beta=("beta_value", "max"),
    )
)

print()
print("Minimal SeSAMe payload-content validation completed.")
print(
    f"Payloads expected:          "
    f"{EXPECTED_FILE_COUNT:,}"
)
print(
    f"Payloads with five samples: "
    f"{(sample_counts_per_payload == 5).sum():,}"
)
print(
    f"Sampled records inspected:  "
    f"{len(sampled_payload_records_df):,}"
)
print(
    f"Validation errors:          "
    f"{len(content_validation_errors_df):,}"
)
print(
    f"Missing sampled betas:      "
    f"{len(missing_sampled_beta_values_df):,}"
)

print()
print("Probe prefixes:")
print(probe_prefix_summary)

print()
print("Sampled beta-value ranges by platform:")
print(sampled_beta_range_by_platform)

if not content_validation_errors_df.empty:
    print()
    print("Content-validation errors:")
    print(
        content_validation_errors_df[
            [
                "file_id",
                "file_name",
                "platform",
                "sample_position",
                "validation_error",
            ]
        ].head(20)
    )


# -----------------------------------------------------------------------------
# Fail closed if minimal content validation is unsuccessful
# -----------------------------------------------------------------------------

content_failures = {
    "payloads_with_unexpected_sample_count": len(
        unexpected_sample_counts_df
    ),
    "sampled_record_validation_errors": len(
        content_validation_errors_df
    ),
}

nonzero_content_failures = {
    label: count
    for label, count in content_failures.items()
    if count != 0
}

if nonzero_content_failures:
    raise RuntimeError(
        "Local payloads failed minimal SeSAMe "
        "content validation:\n"
        + json.dumps(
            nonzero_content_failures,
            indent=2,
        )
    )

print()
print(
    "All local methylation payloads passed "
    "minimal SeSAMe content validation."
)

Inspected 2,000 / 10,897 payloads
Inspected 4,000 / 10,897 payloads
Inspected 6,000 / 10,897 payloads
Inspected 8,000 / 10,897 payloads
Inspected 10,000 / 10,897 payloads
Inspected 10,897 / 10,897 payloads

Minimal SeSAMe payload-content validation completed.
Payloads expected:          10,897
Payloads with five samples: 10,897
Sampled records inspected:  54,485
Validation errors:          0
Missing sampled betas:      13,375

Probe prefixes:
  probe_prefix  sampled_record_count
0           cg                 45867
1           rs                  8618

Sampled beta-value ranges by platform:
                         platform  sampled_records  minimum_beta  maximum_beta
0   Illumina Human Methylation 27            10838      0.003721      0.989242
1  Illumina Human Methylation 450            30272      0.007364      0.992881

All local methylation payloads passed minimal SeSAMe content validation.


In [10]:
# =============================================================================
# Characterize sampled missing beta values and probe-prefix patterns
# =============================================================================

sampled_content_audit_df = (
    sampled_payload_records_df
    .copy()
)

sampled_content_audit_df["has_numeric_beta"] = (
    sampled_content_audit_df["beta_value"].notna()
)

sampled_content_audit_df["has_missing_beta"] = (
    sampled_content_audit_df["beta_is_missing"]
)


# -----------------------------------------------------------------------------
# Missing beta values by platform
# -----------------------------------------------------------------------------

missing_beta_by_platform = (
    sampled_content_audit_df
    .groupby("platform", as_index=False)
    .agg(
        payload_count=("file_id", "nunique"),
        sampled_records=("file_id", "size"),
        numeric_beta_records=(
            "has_numeric_beta",
            "sum",
        ),
        missing_beta_records=(
            "has_missing_beta",
            "sum",
        ),
    )
)

missing_beta_by_platform[
    "missing_beta_percent"
] = (
    100
    * missing_beta_by_platform[
        "missing_beta_records"
    ]
    / missing_beta_by_platform[
        "sampled_records"
    ]
)


# -----------------------------------------------------------------------------
# Missing beta values by platform and sampled position
# -----------------------------------------------------------------------------

missing_beta_by_position = (
    sampled_content_audit_df
    .groupby(
        [
            "platform",
            "sample_position",
        ],
        as_index=False,
    )
    .agg(
        sampled_records=("file_id", "size"),
        numeric_beta_records=(
            "has_numeric_beta",
            "sum",
        ),
        missing_beta_records=(
            "has_missing_beta",
            "sum",
        ),
    )
)

missing_beta_by_position[
    "missing_beta_percent"
] = (
    100
    * missing_beta_by_position[
        "missing_beta_records"
    ]
    / missing_beta_by_position[
        "sampled_records"
    ]
)


# -----------------------------------------------------------------------------
# Number of missing sampled betas per payload
# -----------------------------------------------------------------------------

missing_beta_per_payload = (
    sampled_content_audit_df
    .groupby(
        [
            "file_id",
            "platform",
        ],
        as_index=False,
    )
    .agg(
        sampled_records=("file_id", "size"),
        numeric_beta_records=(
            "has_numeric_beta",
            "sum",
        ),
        missing_beta_records=(
            "has_missing_beta",
            "sum",
        ),
    )
)

missing_count_distribution = (
    missing_beta_per_payload
    .groupby(
        [
            "platform",
            "missing_beta_records",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "payload_count"})
)

payloads_without_numeric_samples_df = (
    missing_beta_per_payload.loc[
        missing_beta_per_payload[
            "numeric_beta_records"
        ] == 0
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Missing-value tokens
# -----------------------------------------------------------------------------

missing_beta_token_summary = (
    sampled_content_audit_df.loc[
        sampled_content_audit_df[
            "has_missing_beta"
        ],
        "beta_token",
    ]
    .value_counts(dropna=False)
    .rename_axis("missing_beta_token")
    .reset_index(name="sampled_record_count")
)


# -----------------------------------------------------------------------------
# Probe prefixes by platform and position
# -----------------------------------------------------------------------------

probe_prefix_by_position = (
    sampled_content_audit_df
    .groupby(
        [
            "platform",
            "sample_position",
            "probe_prefix",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "sampled_record_count"})
)


# -----------------------------------------------------------------------------
# Audit summary
# -----------------------------------------------------------------------------

print("Sampled beta-value missingness characterized.")

print()
print("Missing beta values by platform:")
print(missing_beta_by_platform)

print()
print("Missing beta values by platform and position:")
print(missing_beta_by_position)

print()
print("Missing sampled beta counts per payload:")
print(missing_count_distribution)

print()
print("Missing-value tokens:")
print(missing_beta_token_summary)

print()
print("Probe prefixes by platform and position:")
print(probe_prefix_by_position)

print()
print(
    "Payloads without any numeric sampled beta value: "
    f"{len(payloads_without_numeric_samples_df):,}"
)

if not payloads_without_numeric_samples_df.empty:
    print(
        payloads_without_numeric_samples_df.head(20)
    )

Sampled beta-value missingness characterized.

Missing beta values by platform:
                         platform  payload_count  sampled_records  \
0   Illumina Human Methylation 27           2279            11395   
1  Illumina Human Methylation 450           8618            43090   

   numeric_beta_records  missing_beta_records  missing_beta_percent  
0                 10838                   557              4.888109  
1                 30272                 12818             29.747041  

Missing beta values by platform and position:
                         platform         sample_position  sampled_records  \
0   Illumina Human Methylation 27            first_record             2279   
1   Illumina Human Methylation 27             last_record             2279   
2   Illumina Human Methylation 27         middle_position             2279   
3   Illumina Human Methylation 27        quarter_position             2279   
4   Illumina Human Methylation 27  three_quarter_position        

In [11]:
# =============================================================================
# Fully profile payloads with no numeric beta values in sampled positions
# =============================================================================

full_profile_targets_df = (
    payloads_without_numeric_samples_df[
        ["file_id"]
    ]
    .merge(
        payload_format_inventory_df[
            [
                "file_id",
                "file_name",
                "platform",
                "project_id",
                "local_path",
                "local_size_bytes",
            ]
        ],
        on="file_id",
        how="left",
        validate="one_to_one",
    )
)

if full_profile_targets_df["local_path"].isna().any():
    raise RuntimeError(
        "Could not resolve every full-profile target "
        "to a local payload."
    )


# -----------------------------------------------------------------------------
# Parse each target payload completely
# -----------------------------------------------------------------------------

full_payload_profile_rows = []

for row in full_profile_targets_df.itertuples(
    index=False
):
    payload_path = Path(row.local_path)

    try:
        payload_frame = pd.read_csv(
            payload_path,
            sep="\t",
            header=None,
            names=[
                "probe_id",
                "beta_token",
            ],
            dtype="string",
            keep_default_na=False,
            skip_blank_lines=False,
        )

        read_error = None

    except Exception as error:
        payload_frame = None
        read_error = (
            f"{type(error).__name__}: {error}"
        )

    if read_error is not None:
        full_payload_profile_rows.append(
            {
                "file_id": row.file_id,
                "file_name": row.file_name,
                "project_id": row.project_id,
                "platform": row.platform,
                "total_records": None,
                "unique_probe_ids": None,
                "duplicate_probe_ids": None,
                "empty_probe_ids": None,
                "cg_probe_records": None,
                "rs_probe_records": None,
                "other_probe_records": None,
                "numeric_beta_records": None,
                "missing_beta_records": None,
                "missing_beta_percent": None,
                "invalid_beta_tokens": None,
                "nonfinite_beta_values": None,
                "out_of_range_beta_values": None,
                "minimum_beta": None,
                "maximum_beta": None,
                "read_error": read_error,
            }
        )

        continue

    probe_ids = (
        payload_frame["probe_id"]
        .str.strip()
    )

    beta_tokens = (
        payload_frame["beta_token"]
        .str.strip()
    )

    missing_beta_mask = (
        beta_tokens
        .str.lower()
        .isin(MISSING_BETA_TOKENS)
    )

    numeric_beta_values = pd.to_numeric(
        beta_tokens.mask(missing_beta_mask),
        errors="coerce",
    )

    invalid_beta_token_mask = (
        ~missing_beta_mask
        & numeric_beta_values.isna()
    )

    nonfinite_beta_mask = (
        numeric_beta_values.eq(float("inf"))
        | numeric_beta_values.eq(float("-inf"))
    )

    out_of_range_beta_mask = (
        numeric_beta_values.notna()
        & ~nonfinite_beta_mask
        & ~numeric_beta_values.between(
            0.0,
            1.0,
            inclusive="both",
        )
    )

    valid_numeric_beta_mask = (
        numeric_beta_values.notna()
        & ~nonfinite_beta_mask
        & ~out_of_range_beta_mask
    )

    probe_prefixes = (
        probe_ids
        .str.extract(
            r"^([A-Za-z]+)",
            expand=False,
        )
        .fillna("")
    )

    total_records = len(payload_frame)
    missing_beta_records = int(
        missing_beta_mask.sum()
    )

    full_payload_profile_rows.append(
        {
            "file_id": row.file_id,
            "file_name": row.file_name,
            "project_id": row.project_id,
            "platform": row.platform,
            "total_records": total_records,
            "unique_probe_ids": int(
                probe_ids.nunique()
            ),
            "duplicate_probe_ids": int(
                probe_ids.duplicated().sum()
            ),
            "empty_probe_ids": int(
                probe_ids.eq("").sum()
            ),
            "cg_probe_records": int(
                probe_prefixes.eq("cg").sum()
            ),
            "rs_probe_records": int(
                probe_prefixes.eq("rs").sum()
            ),
            "other_probe_records": int(
                (
                    ~probe_prefixes.isin(
                        ["cg", "rs"]
                    )
                ).sum()
            ),
            "numeric_beta_records": int(
                valid_numeric_beta_mask.sum()
            ),
            "missing_beta_records": (
                missing_beta_records
            ),
            "missing_beta_percent": (
                100
                * missing_beta_records
                / total_records
                if total_records > 0
                else None
            ),
            "invalid_beta_tokens": int(
                invalid_beta_token_mask.sum()
            ),
            "nonfinite_beta_values": int(
                nonfinite_beta_mask.sum()
            ),
            "out_of_range_beta_values": int(
                out_of_range_beta_mask.sum()
            ),
            "minimum_beta": (
                numeric_beta_values.loc[
                    valid_numeric_beta_mask
                ].min()
                if valid_numeric_beta_mask.any()
                else None
            ),
            "maximum_beta": (
                numeric_beta_values.loc[
                    valid_numeric_beta_mask
                ].max()
                if valid_numeric_beta_mask.any()
                else None
            ),
            "read_error": None,
        }
    )


full_payload_profiles_df = pd.DataFrame(
    full_payload_profile_rows
)


# -----------------------------------------------------------------------------
# Display complete profiles
# -----------------------------------------------------------------------------

print(
    "Full profiles for payloads without "
    "numeric sampled beta values:"
)
print()

with pd.option_context(
    "display.max_columns",
    None,
    "display.width",
    240,
):
    print(full_payload_profiles_df)


# -----------------------------------------------------------------------------
# Validate that the target payloads contain usable numeric data
# -----------------------------------------------------------------------------

full_profile_failures = {
    "read_errors": int(
        full_payload_profiles_df[
            "read_error"
        ].notna().sum()
    ),
    "payloads_without_numeric_betas": int(
        full_payload_profiles_df[
            "numeric_beta_records"
        ]
        .fillna(0)
        .eq(0)
        .sum()
    ),
    "empty_probe_ids": int(
        full_payload_profiles_df[
            "empty_probe_ids"
        ]
        .fillna(0)
        .sum()
    ),
    "invalid_beta_tokens": int(
        full_payload_profiles_df[
            "invalid_beta_tokens"
        ]
        .fillna(0)
        .sum()
    ),
    "nonfinite_beta_values": int(
        full_payload_profiles_df[
            "nonfinite_beta_values"
        ]
        .fillna(0)
        .sum()
    ),
    "out_of_range_beta_values": int(
        full_payload_profiles_df[
            "out_of_range_beta_values"
        ]
        .fillna(0)
        .sum()
    ),
}

nonzero_full_profile_failures = {
    label: count
    for label, count in full_profile_failures.items()
    if count != 0
}

if nonzero_full_profile_failures:
    raise RuntimeError(
        "One or more fully profiled payloads failed "
        "content validation:\n"
        + json.dumps(
            nonzero_full_profile_failures,
            indent=2,
        )
    )

print()
print(
    "All fully profiled payloads contain valid "
    "numeric beta values and passed content validation."
)

Full profiles for payloads without numeric sampled beta values:

                                file_id                                          file_name project_id                        platform  total_records  unique_probe_ids  duplicate_probe_ids  empty_probe_ids  cg_probe_records  \
0  bc9f178a-9c93-4f5b-8c6b-4a4a44b10716  d3e86a50-4a4a-4c8d-8e18-e0b0ee7ecc6b.methylati...  TCGA-CESC  Illumina Human Methylation 450         486427            486427                    0                0            482421   
1  bd2f9497-2542-42ea-b36b-7649987ab1fd  b91998ca-c51b-4ca0-94bb-9af542c38812.methylati...  TCGA-LIHC  Illumina Human Methylation 450         486427            486427                    0                0            482421   
2  ef2ec064-4dbd-4de8-bcf0-4323916828b7  5a84a3ff-a6eb-482d-a5c0-b9f850737f2e.methylati...  TCGA-LIHC  Illumina Human Methylation 450         486427            486427                    0                0            482421   

   rs_probe_records  other_pro

In [12]:
# =============================================================================
# Parse and audit GDC annotation records
# =============================================================================

EXPECTED_ANNOTATION_COLUMNS = [
    "id",
    "submitter_id",
    "entity_type",
    "entity_id",
    "category",
    "classification",
    "created_datetime",
    "status",
    "notes",
]


# -----------------------------------------------------------------------------
# Parse every annotations.txt file
# -----------------------------------------------------------------------------

parsed_annotation_frames = []

for row in (
    annotation_files_df
    .sort_values(["folder_id", "file_name"])
    .itertuples(index=False)
):
    annotation_path = Path(row.file_path)

    try:
        annotation_frame = pd.read_csv(
            annotation_path,
            sep="\t",
            dtype="string",
            keep_default_na=False,
        )

    except Exception as error:
        raise RuntimeError(
            f"Could not parse annotation file: "
            f"{annotation_path}"
        ) from error

    observed_columns = annotation_frame.columns.tolist()

    if observed_columns != EXPECTED_ANNOTATION_COLUMNS:
        raise ValueError(
            "Unexpected annotation-file schema:\n"
            f"File: {annotation_path}\n"
            f"Observed: {observed_columns}\n"
            f"Expected: {EXPECTED_ANNOTATION_COLUMNS}"
        )

    if annotation_frame.empty:
        raise ValueError(
            f"Annotation file contains no records: "
            f"{annotation_path}"
        )

    annotation_frame.insert(
        0,
        "file_id",
        str(row.folder_id),
    )

    annotation_frame["annotation_path"] = annotation_path

    parsed_annotation_frames.append(annotation_frame)


annotation_associations_df = pd.concat(
    parsed_annotation_frames,
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Validate file-to-annotation associations
# -----------------------------------------------------------------------------

duplicate_annotation_associations_df = (
    annotation_associations_df.loc[
        annotation_associations_df.duplicated(
            subset=["file_id", "id"],
            keep=False,
        )
    ]
    .sort_values(["file_id", "id"])
    .copy()
)

annotation_metadata_columns = [
    column
    for column in EXPECTED_ANNOTATION_COLUMNS
    if column != "id"
]

annotation_metadata_variation = (
    annotation_associations_df
    .groupby("id", dropna=False)[
        annotation_metadata_columns
    ]
    .nunique(dropna=False)
)

conflicting_annotation_ids = (
    annotation_metadata_variation
    .gt(1)
    .any(axis=1)
)

conflicting_annotation_ids = set(
    annotation_metadata_variation.index[
        conflicting_annotation_ids
    ]
)

conflicting_annotations_df = (
    annotation_associations_df.loc[
        annotation_associations_df["id"].isin(
            conflicting_annotation_ids
        )
    ]
    .sort_values(["id", "file_id"])
    .copy()
)


# -----------------------------------------------------------------------------
# Build unique annotation table
# -----------------------------------------------------------------------------

unique_annotations_df = (
    annotation_associations_df[
        EXPECTED_ANNOTATION_COLUMNS
    ]
    .drop_duplicates()
    .sort_values("id")
    .reset_index(drop=True)
)

if unique_annotations_df["id"].duplicated().any():
    raise RuntimeError(
        "Annotation IDs remain duplicated after exact-row "
        "deduplication, indicating conflicting metadata."
    )


# -----------------------------------------------------------------------------
# Check required identifier fields
# -----------------------------------------------------------------------------

required_nonempty_annotation_fields = [
    "id",
    "entity_type",
    "entity_id",
    "category",
    "classification",
    "created_datetime",
    "status",
]

empty_required_field_counts = {
    column: int(
        unique_annotations_df[column]
        .str.strip()
        .eq("")
        .sum()
    )
    for column in required_nonempty_annotation_fields
}

nonzero_empty_required_fields = {
    column: count
    for column, count in empty_required_field_counts.items()
    if count != 0
}


# -----------------------------------------------------------------------------
# Summary tables
# -----------------------------------------------------------------------------

annotations_per_file_distribution = (
    annotation_associations_df
    .groupby("file_id")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("annotations_per_file")
    .reset_index(name="file_count")
)

files_per_annotation_distribution = (
    annotation_associations_df
    .groupby("id")["file_id"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("files_per_annotation")
    .reset_index(name="annotation_count")
)

entity_type_summary = (
    unique_annotations_df["entity_type"]
    .value_counts()
    .rename_axis("entity_type")
    .reset_index(name="annotation_count")
)

classification_summary = (
    unique_annotations_df["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="annotation_count")
)

status_summary = (
    unique_annotations_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="annotation_count")
)

category_summary = (
    unique_annotations_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="annotation_count")
)


# -----------------------------------------------------------------------------
# Annotation audit summary
# -----------------------------------------------------------------------------

print("GDC annotation records parsed and audited.")
print(
    f"Annotation files parsed:          "
    f"{len(annotation_files_df):,}"
)
print(
    f"File-to-annotation associations:  "
    f"{len(annotation_associations_df):,}"
)
print(
    f"Unique annotation IDs:            "
    f"{len(unique_annotations_df):,}"
)
print(
    f"Unique annotated entity IDs:      "
    f"{unique_annotations_df['entity_id'].nunique():,}"
)
print(
    f"Duplicate file-annotation links:  "
    f"{len(duplicate_annotation_associations_df):,}"
)
print(
    f"Conflicting annotation IDs:       "
    f"{len(conflicting_annotation_ids):,}"
)
print(
    f"Empty required fields:            "
    f"{sum(nonzero_empty_required_fields.values()):,}"
)

print()
print("Annotations per payload file:")
print(annotations_per_file_distribution)

print()
print("Payload files per unique annotation:")
print(files_per_annotation_distribution)

print()
print("Entity types:")
print(entity_type_summary)

print()
print("Classifications:")
print(classification_summary)

print()
print("Statuses:")
print(status_summary)

print()
print("Annotation categories:")
print(category_summary)


# -----------------------------------------------------------------------------
# Fail closed on malformed or contradictory annotation metadata
# -----------------------------------------------------------------------------

annotation_failures = {
    "duplicate_file_annotation_links": len(
        duplicate_annotation_associations_df
    ),
    "conflicting_annotation_ids": len(
        conflicting_annotation_ids
    ),
    "empty_required_fields": sum(
        nonzero_empty_required_fields.values()
    ),
}

nonzero_annotation_failures = {
    label: count
    for label, count in annotation_failures.items()
    if count != 0
}

if nonzero_annotation_failures:
    raise RuntimeError(
        "GDC annotation metadata failed validation:\n"
        + json.dumps(
            nonzero_annotation_failures,
            indent=2,
        )
    )

print()
print(
    "All GDC annotation records were parsed without "
    "schema errors, duplicate links, or conflicting metadata."
)

GDC annotation records parsed and audited.
Annotation files parsed:          1,245
File-to-annotation associations:  1,456
Unique annotation IDs:            1,242
Unique annotated entity IDs:      1,051
Duplicate file-annotation links:  0
Conflicting annotation IDs:       0
Empty required fields:            0

Annotations per payload file:
   annotations_per_file  file_count
0                     1        1062
1                     2         158
2                     3          22
3                     4           3

Payload files per unique annotation:
   files_per_annotation  annotation_count
0                     1              1038
1                     2               196
2                     3                 6
3                     4                 2

Entity types:
  entity_type  annotation_count
0        case              1141
1     analyte                87
2     aliquot                 9
3      sample                 4
4     portion                 1

Classifications:
  cla

In [13]:
# =============================================================================
# Save consolidated GDC methylation annotation index
# =============================================================================

METHYLATION_METADATA_OUTPUT_DIR = (
    Paths.interim / "metadata"
)

METHYLATION_ANNOTATION_INDEX_PATH = (
    METHYLATION_METADATA_OUTPUT_DIR
    / "gdc_annotation_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv"
)


# -----------------------------------------------------------------------------
# Build downstream-consumable annotation index
# -----------------------------------------------------------------------------

annotation_index_output_df = (
    annotation_associations_df[
        [
            "file_id",
            "id",
            "submitter_id",
            "entity_type",
            "entity_id",
            "category",
            "classification",
            "created_datetime",
            "status",
            "notes",
        ]
    ]
    .rename(
        columns={
            "id": "annotation_id",
            "submitter_id": "annotation_submitter_id",
        }
    )
    .sort_values(
        [
            "file_id",
            "annotation_id",
        ]
    )
    .reset_index(drop=True)
)

if annotation_index_output_df.duplicated(
    subset=[
        "file_id",
        "annotation_id",
    ]
).any():
    raise RuntimeError(
        "Duplicate file-to-annotation associations "
        "remain in the output table."
    )

if (
    annotation_index_output_df[
        "annotation_id"
    ].nunique()
    != len(unique_annotations_df)
):
    raise RuntimeError(
        "The output annotation index does not preserve "
        "all unique GDC annotation IDs."
    )


# -----------------------------------------------------------------------------
# Write and validate the output
# -----------------------------------------------------------------------------

METHYLATION_METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

annotation_index_output_df.to_csv(
    METHYLATION_ANNOTATION_INDEX_PATH,
    sep="\t",
    index=False,
    encoding="utf-8",
    lineterminator="\n",
)

reloaded_annotation_index_df = pd.read_csv(
    METHYLATION_ANNOTATION_INDEX_PATH,
    sep="\t",
    dtype="string",
    keep_default_na=False,
)

if (
    reloaded_annotation_index_df.shape
    != annotation_index_output_df.shape
):
    raise RuntimeError(
        "Saved annotation index failed shape validation."
    )

if (
    reloaded_annotation_index_df.columns.tolist()
    != annotation_index_output_df.columns.tolist()
):
    raise RuntimeError(
        "Saved annotation index failed schema validation."
    )

if reloaded_annotation_index_df.duplicated(
    subset=[
        "file_id",
        "annotation_id",
    ]
).any():
    raise RuntimeError(
        "Saved annotation index contains duplicate "
        "file-to-annotation associations."
    )


# -----------------------------------------------------------------------------
# Output summary
# -----------------------------------------------------------------------------

annotation_index_sha256 = calculate_sha256(
    METHYLATION_ANNOTATION_INDEX_PATH
)

print("Consolidated GDC annotation index written.")
print(
    f"Output path:          "
    f"{to_project_relative_posix_path(METHYLATION_ANNOTATION_INDEX_PATH, PROJECT_ROOT)}"
)
print(
    f"File size:            "
    f"{get_file_size_mb(METHYLATION_ANNOTATION_INDEX_PATH):,.3f} MiB"
)
print(
    f"Association rows:     "
    f"{len(annotation_index_output_df):,}"
)
print(
    f"Unique file IDs:      "
    f"{annotation_index_output_df['file_id'].nunique():,}"
)
print(
    f"Unique annotations:   "
    f"{annotation_index_output_df['annotation_id'].nunique():,}"
)
print(
    f"Unique entities:      "
    f"{annotation_index_output_df['entity_id'].nunique():,}"
)
print(f"SHA-256:              {annotation_index_sha256}")

Consolidated GDC annotation index written.
Output path:          data/interim/metadata/gdc_annotation_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv
File size:            0.452 MiB
Association rows:     1,456
Unique file IDs:      1,245
Unique annotations:   1,242
Unique entities:      1,051
SHA-256:              cfdd68a3bd9f51f9c79e0345b3227ec976daea787b0bd8567557ae1b49b5167a


## Summary

The downloaded TCGA primary-tumor DNA methylation cohort was validated against the frozen HM27/HM450 acquisition artifacts generated in notebooks 104 and 105.

### Acquisition integrity

- The frozen cohort contains 10,897 SeSAMe methylation beta-value payloads:
  - 2,279 Illumina Human Methylation 27 files.
  - 8,618 Illumina Human Methylation 450 files.
- All 10,897 expected file-ID directories were present.
- All 10,897 expected payload filenames were present in their corresponding directories.
- No expected or unexpected file-ID directories were detected.
- No payloads were missing.
- No payload size mismatches were detected.
- No unexpected auxiliary files were present.
- No direct or nested partial-download files remained.
- The observed payload size exactly matched the frozen manifest: 106.735678 GiB.
- All 10,897 local payload MD5 checksums matched the frozen GDC manifest.
- No payload read errors or checksum mismatches were detected.

These results establish that the local methylation payloads are byte-identical to the files specified by the frozen GDC manifest.

### Auxiliary download content

The local download additionally contained:

- 1,245 `annotations.txt` files comprising 1,456 file-to-annotation associations.
- 1,242 unique GDC annotation IDs associated with 1,051 unique entities.
- 9,102 `logs/` directories containing 9,102 nested log files.

All annotation files were readable and followed the expected nine-column GDC tabular schema. No duplicate file-to-annotation links, conflicting annotation metadata, or empty required fields were detected. All unique annotations had `Approved` status.

Annotation categories included prior malignancy, alternate sample pipeline, neoadjuvant therapy, protocol deviations, noncanonical items, consent withdrawal, qualification errors, and other potentially QC-relevant metadata. These records were preserved without applying exclusion decisions at the acquisition-validation stage.

### Minimal payload-content validation

Representative HM27 and HM450 payloads confirmed that the SeSAMe files are headerless, two-column, tab-delimited text files containing probe identifiers and beta values.

Five deterministic positions were inspected in every payload, producing 54,485 sampled records:

- All 10,897 payloads yielded the five expected sampled records.
- No encoding, structural, numeric-conversion, nonfinite-value, or beta-range errors were detected.
- Probe prefixes included `cg` and `rs`.
- All observed numeric beta values fell within the expected `[0, 1]` interval.
- Missing beta values were consistently encoded as `NA`.

The terminal HM450 record corresponded to an `rs` probe and was `NA` in 8,616 of 8,618 HM450 payloads, substantially increasing the missingness estimate produced by the deterministic sampling scheme.

Three HM450 payloads had no numeric beta values among the five sampled positions and were therefore parsed completely. Each contained 486,427 unique probe records, no duplicated or empty probe identifiers, and between 375,290 and 395,690 valid numeric beta values. Their complete-file missingness ranged from 18.65% to 22.85%, and no malformed, nonfinite, or out-of-range beta values were detected.

### Downstream metadata output

A consolidated file-to-annotation index was written to:

`data/interim/metadata/gdc_annotation_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv`

The output contains:

- 1,456 file-to-annotation associations.
- 1,245 annotated methylation file IDs.
- 1,242 unique annotation IDs.
- 1,051 unique annotated entities.
- SHA-256: `cfdd68a3bd9f51f9c79e0345b3227ec976daea787b0bd8567557ae1b49b5167a`.

This table is retained as a downstream metadata and quality-control input. No other diagnostic tables from this notebook were persisted.

### Scope and conclusion

The frozen HM27/HM450 acquisition cohort passed structural, size, checksum, auxiliary-metadata, and minimal content-level validation. The local download is complete and suitable for downstream methylation quality-control processing.

This notebook does not perform sample exclusion, probe filtering, platform harmonization, normalization, or biological quality control. GDC annotation categories and payload-level missingness remain metadata to be evaluated explicitly in downstream QC analyses.